# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv('ames_housing.csv', encoding='utf-8-sig')

print('데이터 크기 :', df.shape)
print('전체 결측치 수 :', df.isnull().sum().sum())
df.head()

데이터 크기 : (1460, 10)
전체 결측치 수 : 0


,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

# OverallQual 5, 6, 7 집단에서 각각 표본 20개 추출
group_5 = df[df['OverallQual'] == 5]['SalePrice'].sample(n=20, random_state=5)
group_6 = df[df['OverallQual'] == 6]['SalePrice'].sample(n=20, random_state=5)
group_7 = df[df['OverallQual'] == 7]['SalePrice'].sample(n=20, random_state=5)


# 1. 기술통계
print('[기술통계]')

print(f'OverallQual 5 - 표본 수 : {len(group_5)} | 평균 : {group_5.mean():.3f} | 표준편차 : {group_5.std():.3f}')
print(f'OverallQual 6 - 표본 수 : {len(group_6)} | 평균 : {group_6.mean():.3f} | 표준편차 : {group_6.std():.3f}')
print(f'OverallQual 7 - 표본 수 : {len(group_7)} | 평균 : {group_7.mean():.3f} | 표준편차 : {group_7.std():.3f}')


# 2. 독립집단 여부
print('\n[독립집단]')
print('OverallQual이 5, 6, 7인 집단은 서로 다른 주택으로 구성되어 있으므로 독립집단이다.')


# 3. Shapiro-Wilk 정규성 검정
print('\n[Shapiro-Wilk 정규성 검정]')

shapiro_5_stat, shapiro_5_p = stats.shapiro(group_5)
shapiro_6_stat, shapiro_6_p = stats.shapiro(group_6)
shapiro_7_stat, shapiro_7_p = stats.shapiro(group_7)

print(f'OverallQual 5 | 검정통계량 : {shapiro_5_stat:.3f} | p-value : {shapiro_5_p:.3f}')
print(f'OverallQual 6 | 검정통계량 : {shapiro_6_stat:.3f} | p-value : {shapiro_6_p:.3f}')
print(f'OverallQual 7 | 검정통계량 : {shapiro_7_stat:.3f} | p-value : {shapiro_7_p:.3f}')

print()

print('OverallQual 5 : 정규성 가정을 만족합니다.' if shapiro_5_p >= 0.05 else 'OverallQual 5 : 정규성 가정을 만족하지 않습니다.')
print('OverallQual 6 : 정규성 가정을 만족합니다.' if shapiro_6_p >= 0.05 else 'OverallQual 6 : 정규성 가정을 만족하지 않습니다.')
print('OverallQual 7 : 정규성 가정을 만족합니다.' if shapiro_7_p >= 0.05 else 'OverallQual 7 : 정규성 가정을 만족하지 않습니다.')


# 4. Levene 등분산 검정
print('\n[Levene 등분산 검정]')

levene_stat, levene_p = stats.levene(group_5, group_6, group_7)

print(f'검정통계량 : {levene_stat:.3f}')
print(f'p-value : {levene_p:.3f}')
print('등분산 가정을 만족합니다.' if levene_p >= 0.05 else '등분산 가정을 만족하지 않습니다.')


# 5. 가설
print('\n[가설]')
print('H0 : 세 집단의 모집단 평균 판매가격은 모두 같다.')
print('H1 : 적어도 한 집단의 모집단 평균 판매가격은 다르다.')


# 6. 일원배치 ANOVA
print('\n[One-Way ANOVA]')

f_stat, p_value = stats.f_oneway(group_5, group_6, group_7)

print(f'F통계량 : {f_stat:.3f}')
print(f'p-value : {p_value:.3f}')

print('귀무가설을 기각합니다. 적어도 한 집단의 평균 판매가격에 차이가 있습니다.'
      if p_value < 0.05
      else '귀무가설을 기각하지 못합니다. 세 집단의 평균 판매가격에 유의한 차이가 있다고 보기 어렵습니다.')


[기술통계]
OverallQual 5 - 표본 수 : 20 | 평균 : 130605.000 | 표준편차 : 24937.110
OverallQual 6 - 표본 수 : 20 | 평균 : 167826.600 | 표준편차 : 41944.554
OverallQual 7 - 표본 수 : 20 | 평균 : 217593.600 | 표준편차 : 48298.393

[독립집단]
OverallQual이 5, 6, 7인 집단은 서로 다른 주택으로 구성되어 있으므로 독립집단이다.

[Shapiro-Wilk 정규성 검정]
OverallQual 5 | 검정통계량 : 0.971 | p-value : 0.776
OverallQual 6 | 검정통계량 : 0.953 | p-value : 0.410
OverallQual 7 | 검정통계량 : 0.926 | p-value : 0.129

OverallQual 5 : 정규성 가정을 만족합니다.
OverallQual 6 : 정규성 가정을 만족합니다.
OverallQual 7 : 정규성 가정을 만족합니다.

[Levene 등분산 검정]
검정통계량 : 2.652
p-value : 0.079
등분산 가정을 만족합니다.

[가설]
H0 : 세 집단의 모집단 평균 판매가격은 모두 같다.
H1 : 적어도 한 집단의 모집단 평균 판매가격은 다르다.

[One-Way ANOVA]
F통계량 : 24.246
p-value : 0.000
귀무가설을 기각합니다. 적어도 한 집단의 평균 판매가격에 차이가 있습니다.


### 필수 1 답변 작성란

- **Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?
    -  검정을 반복할수록 전체 분서에서 한 번 이상 제 1종 오류가 발생확률이 0.05보다 커지는 다중 비교 문제가 발생한다.

- **Q2.** F통계량은 어떤 두 변동의 비율인가요?  
    - 집단 간 평균 차이를 나타내는 집단 간 변동을 같은 집단 내부의 개인차인 집단 내 변동으로 나눈 비율

- **Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
    - 유의미한 차이가 있다. 따라서 적어도 한 집단의 모집간 평균 판매가격은 다르다. 

- **Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?
    - 알 수 없다. 구체적인 집단 쌍은 Tukey HSD와 같은 사후 검정으로 확인해야한다.

---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [3]:
# 필수 2 코드를 작성하세요.

import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations


# 1. 세 집단을 하나의 데이터프레임으로 결합
anova_df = pd.concat([
    pd.DataFrame({'OverallQual': 5, 'SalePrice': group_5.values}),
    pd.DataFrame({'OverallQual': 6, 'SalePrice': group_6.values}),
    pd.DataFrame({'OverallQual': 7, 'SalePrice': group_7.values})
], ignore_index=True)

print('[anova_df]')
display(anova_df.head())


# 2. 전체 평균
grand_mean = anova_df['SalePrice'].mean()

print(f'\n전체 평균 : {grand_mean:.3f}')


# 집단별 평균과 표본 수
group_mean = anova_df.groupby('OverallQual')['SalePrice'].mean()
group_count = anova_df.groupby('OverallQual')['SalePrice'].count()


# 3. ANOVA 직접 계산

# 집단 간 제곱합
SS_between = sum(
    group_count[i] * (group_mean[i] - grand_mean) ** 2
    for i in group_mean.index
)

# 집단 내 제곱합
SS_within = sum(
    ((anova_df[anova_df['OverallQual'] == i]['SalePrice'] - group_mean[i]) ** 2).sum()
    for i in group_mean.index
)

# 집단 수와 전체 표본 수
k = anova_df['OverallQual'].nunique()
N = len(anova_df)

# 자유도
df_between = k - 1
df_within = N - k

# 평균제곱
MS_between = SS_between / df_between
MS_within = SS_within / df_within

# F통계량
F_stat_manual = MS_between / MS_within


# ANOVA 표
anova_table = pd.DataFrame({
    '제곱합(SS)': [SS_between, SS_within],
    '자유도(df)': [df_between, df_within],
    '평균제곱(MS)': [MS_between, MS_within],
    'F통계량': [F_stat_manual, None]
}, index=['집단 간', '집단 내'])

print('\n[ANOVA 표]')
display(anova_table.round(3))


# 4. stats.f_oneway()와 비교
F_stat, p_value = stats.f_oneway(group_5, group_6, group_7)

print('\n[F통계량 비교]')
print(f'직접 계산 F통계량 : {F_stat_manual:.3f}')
print(f'f_oneway F통계량 : {F_stat:.3f}')
print(f'p-value : {p_value:.3f}')

print('F통계량이 일치합니다.'
      if round(F_stat_manual, 3) == round(F_stat, 3)
      else 'F통계량을 다시 확인하세요.')


# 5. p-value가 0.05보다 작을 때만 Tukey 사후검정
if p_value < 0.05:

    print('\n[Tukey 사후검정]')

    tukey = pairwise_tukeyhsd(
        endog=anova_df['SalePrice'],
        groups=anova_df['OverallQual'],
        alpha=0.05
    )

    print(tukey)


    # 6. reject=True인 집단 쌍 확인
    print('\n[유의한 차이가 있는 집단 쌍]')

    pairs = list(combinations(tukey.groupsunique, 2))

    for pair, reject in zip(pairs, tukey.reject):
        if reject:
            print(f'{pair[0]} - {pair[1]} : reject=True')


    # 7. 집단 쌍별 평균 차이
    print('\n[집단별 평균]')
    print(group_mean.round(3))

    print('\n[집단 쌍별 평균 차이]')

    for g1, g2 in combinations(group_mean.index, 2):
        mean_diff = group_mean[g2] - group_mean[g1]

        print(
            f'OverallQual {g1} → {g2} : '
            f'{group_mean[g2]:.3f} - {group_mean[g1]:.3f} '
            f'= {mean_diff:.3f}'
        )

else:
    print('\np-value가 0.05 이상이므로 Tukey 사후검정을 수행하지 않습니다.')

[anova_df]


,OverallQual,SalePrice
0,5,132500
1,5,120000
2,5,110000
3,5,172500
4,5,144500



전체 평균 : 172008.400

[ANOVA 표]


,제곱합(SS),자유도(df),평균제곱(MS),F통계량
집단 간,7.619479e+10,2,3.809739e+10,24.246
집단 내,8.956486e+10,57,1.571313e+09,NaN



[F통계량 비교]
직접 계산 F통계량 : 24.246
f_oneway F통계량 : 24.246
p-value : 0.000
F통계량이 일치합니다.

[Tukey 사후검정]
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414   True
-----------------------------------------------------------

[유의한 차이가 있는 집단 쌍]
5 - 6 : reject=True
5 - 7 : reject=True
6 - 7 : reject=True

[집단별 평균]
OverallQual
5    130605.0
6    167826.6
7    217593.6
Name: SalePrice, dtype: float64

[집단 쌍별 평균 차이]
OverallQual 5 → 6 : 167826.600 - 130605.000 = 37221.600
OverallQual 5 → 7 : 217593.600 - 130605.000 = 86988.600
OverallQual 6 → 7 : 217593.600 - 167826.600 = 49767.000


### 필수 2 답변 작성란

- **Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
    - SS_between은 집단 평균들이 전체 평균에서 벗어난 집단 간 변동 값
    - SS_within은 각 관측값이 소속집단 평균에서 벗어난 집단 내 변동

- **Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
    - 다중비교 오류를 조정한 뒤에도 해당 두 집단의 평균이 같다는 귀무가설을 기각하겠다는 의미

- **Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
    - `5 - 6`, `5 - 7`, `6 - 7`의 모든 집단 쌍에서 유의미한 차이가 확인된다. 

- **Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
    - `품질 5-7점 집단`이며 평균 차이는 약 86,988달러이다.

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [4]:
# 과제 코드를 작성하세요.

group_ex = df[df['KitchenQual'] == 'Ex']['SalePrice'].sample(n=20, random_state=18)
group_gd = df[df['KitchenQual'] == 'Gd']['SalePrice'].sample(n=20, random_state=18)
group_ta = df[df['KitchenQual'] == 'TA']['SalePrice'].sample(n=20, random_state=18)

# 1. 기술통계
print('[기술통계]')

print(f'KitehenQual EX - 표본 수 : {len(group_ex)} | 평균 : {group_ex.mean():.3f}')
print(f'KitchenQual Gd - 표본 수 : {len(group_gd)} | 평균 : {group_gd.mean():.3f}')
print(f'KitchenQual TA - 표본 수 : {len(group_ta)} | 평균 : {group_ta.mean():.3f}')

# 2. Shapiro-Wilk 정규성 검정
print('\n[Shapiro-Wilk 정규성 검정]')

shapiro_ex_stat, shapiro_ex_p = stats.shapiro(group_ex)
shapiro_gd_stat, shapiro_gd_p = stats.shapiro(group_gd)
shapiro_ta_stat, shapiro_ta_p = stats.shapiro(group_ta)

print(f'KitchenQual Ex | 검정통계량 : {shapiro_ex_stat:.3f} | p-value : {shapiro_ex_p:.3f}')
print(f'KitchenQual Gd | 검정통계량 : {shapiro_gd_stat:.3f} | p-value : {shapiro_gd_p:.3f}')
print(f'KitchenQual TA | 검정통계량 : {shapiro_ta_stat:.3f} | p-value : {shapiro_ta_p:.3f}')


print('KitchenQual Ex : 정규성 가정을 만족합니다.' if shapiro_ex_p >= 0.05 else 'KitchenQual Ex : 정규성 가정을 만족하지 않습니다.')
print('KitchenQual Gd : 정규성 가정을 만족합니다.' if shapiro_gd_p >= 0.05 else 'KitchenQual Gd : 정규성 가정을 만족하지 않습니다.')
print('KitchenQual TA : 정규성 가정을 만족합니다.' if shapiro_ta_p >= 0.05 else 'KitchenQual TA : 정규성 가정을 만족하지 않습니다.')


# 3. Levene 등분산 검정
print('\n[Levene 등분산 검정]')

levene_stat, levene_p = stats.levene(group_ex, group_gd, group_ta)

print(f'검정통계량 : {levene_stat:.3f}')
print(f'p-value : {levene_p:.3f}')
print('등분산 가정을 만족합니다.' if levene_p >= 0.05 else '등분산 가정을 만족하지 않습니다.')

# 4. 일원배치 ANOVA
print('\n[One-Way ANOVA]')

f_stat, p_value = stats.f_oneway(group_ex, group_gd, group_ta)

print(f'F통계량 : {f_stat:.3f}')
print(f'p-value : {p_value:.3f}')


if p_value <= 0.05:
      print('귀무가설을 기각합니다. 적어도 한 집단의 평균 판매가격에 차이가 있습니다.')

      # 5. Tukey HSD 사후검정
      anova_df = pd.concat([
            pd.DataFrame({'KitchenQual': 'Ex', 'SalePrice': group_ex.values}),
            pd.DataFrame({'KitchenQual': 'Gd', 'SalePrice': group_gd.values}),
            pd.DataFrame({'KitchenQual': 'TA', 'SalePrice': group_ta.values})
      ], ignore_index=True)

      tukey = pairwise_tukeyhsd(
            endog=anova_df['SalePrice'],
            groups=anova_df['KitchenQual'],
            alpha=0.05
    )

      print('\n[Tukey HSD 사후검정]')
      print(tukey)      

      # 6. 유의한 집단 쌍 확인
      print('\n[유의한 집단 쌍]')

      pairs = [('Ex', 'Gd'), ('Ex', 'TA'), ('Gd', 'TA')]

      for pair, reject in zip(pairs, tukey.reject):
            if reject:
                  print(f'{pair[0]} - {pair[1]} : reject=True.')
else:
    print('귀무가설을 기각하지 못합니다. 세 집단의 평균 판매가격에 유의한 차이가 있다고 보기 어렵습니다.')

[기술통계]
KitehenQual EX - 표본 수 : 20 | 평균 : 313983.050
KitchenQual Gd - 표본 수 : 20 | 평균 : 188835.000
KitchenQual TA - 표본 수 : 20 | 평균 : 137486.600

[Shapiro-Wilk 정규성 검정]
KitchenQual Ex | 검정통계량 : 0.967 | p-value : 0.688
KitchenQual Gd | 검정통계량 : 0.972 | p-value : 0.802
KitchenQual TA | 검정통계량 : 0.952 | p-value : 0.397
KitchenQual Ex : 정규성 가정을 만족합니다.
KitchenQual Gd : 정규성 가정을 만족합니다.
KitchenQual TA : 정규성 가정을 만족합니다.

[Levene 등분산 검정]
검정통계량 : 2.555
p-value : 0.087
등분산 가정을 만족합니다.

[One-Way ANOVA]
F통계량 : 54.800
p-value : 0.000
귀무가설을 기각합니다. 적어도 한 집단의 평균 판매가격에 차이가 있습니다.

[Tukey HSD 사후검정]
      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
group1 group2  meandiff  p-adj     lower        upper     reject
----------------------------------------------------------------
    Ex     Gd -125148.05    0.0 -166883.2108  -83412.8892   True
    Ex     TA -176496.45    0.0 -218231.6108 -134761.2892   True
    Gd     TA   -51348.4 0.0122  -93083.5608   -9613.2392   True
---------------------------------

### 과제 답변 작성란

- **Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  

    - p-value가 0.000으로 나타나 유의수준인 0.05보다 작기 때문에 

    - 세 집단의 평균 판매가격에는 유의미한 차이가 있다.

<br>

- **Q2.** 사후 검정은 어떤 조건에서 수행하나요?  

    - 사후검정은 분산분석에서 귀무가설이 기각되고, 최소 한 집단의 평균에 차이가 존재할 때 수행한다.

<br>

- **Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  

    - `EX-GD`, `EX-TA`, `Gd-TA`의 모든 집단에서 유의미한 차이가 발견되었다.

<br>

- **Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

    - 평균 판매가격 차이가 가장 큰 집단 쌍은 `Ex-TA`이며
    
    - 차이는 약 176,496.45원으로 나타난다.

---

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
- 각 검정마다 위양성 가능성이 있기 때문에 비교 횟수가 늘어날수록 전체 분석에서 한 번 이상 잘못 기각될 확률이 누적된다. 

2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
- 귀무가설은 모든 집단의 평균이 같다.
- 대립가설은 적어도 한 집단의 평균이 다르다라는 것

3. F통계량이 크다는 것은 무엇을 의미하나요?
- 집단 내 변동에 비해 집단 간 평균 차이로 설명되는 변동이 상대적으로 크다는 의미

4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
- 적어도 한 집단이 다르다는 사실만 알려주며 구체적인 집단 쌍은 알려주지 않음

5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
- 비교한 집단 쌍, 평균 차이, 조정된 p-value, 신뢰구간, reject 여부